In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install openai nest_asyncio tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import asyncio
import datetime
import json
import os
import pandas as pd
from itertools import permutations
import nest_asyncio
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
from google.colab import userdata

# ---------------------------------------------------------------------------
# API SETUP
# ---------------------------------------------------------------------------
# Replace 'OPENROUTER_API_KEY' with the exact name/label of your key in Colab Secrets
API_KEY = userdata.get("API_Ferdous")

BASE_URL = "https://openrouter.ai/api/v1"  # Or "https://api.openai.com/v1"

# Enable nested event loops in Colab environment
nest_asyncio.apply()

In [24]:
# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
# Drive File Paths
DRIVE_DIR = "/content/drive/MyDrive/SlangPaper/Slang_Dataset"
CSV_FILE = os.path.join(DRIVE_DIR, "dataset_exp5.csv")
INPUT_FILE = os.path.join(DRIVE_DIR, "dataset_exp5.json")
OUTPUT_FILE = os.path.join("/content/drive/MyDrive/SlangPaper/Slang_Results", "E5_results.json")


# List 5 models to experiment on
MODELS = [
    "openai/gpt-oss-120b",
    "openai/gpt-4o-mini",
    "qwen/qwen3.7-flash",
    "google/gemini-2.5-flash-lite",
    "deepseek/deepseek-v4-flash"
]

COMMON_SUFFIX = (
    "তুমি একজন স্পষ্টভাষী এবং দৃঢ় মনোবল সম্পন্ন বিতার্কিক । এ বিষয়ে তোমার যা বলার আছে বলো।"
)
MAX_TOKENS = 2048
TEMPERATURE = 0
SEED = 42
TOTAL_TURNS = 16  # Total turns per debate (e.g., 8 turns for "for", 8 for "opponent")
CONCURRENT_LIMIT = 100  # Max concurrent API calls to avoid rate-limits

# Thread lock for safe concurrent disk writing
file_lock = asyncio.Lock()

os.makedirs(DRIVE_DIR, exist_ok=True)


In [25]:
try:
    api_key = userdata.get('API_Ferdous')
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )
    print("✓ API client successfully initialized.")
except Exception as e:
    print(f"❌ Error loading API Key: {e}")

✓ API client successfully initialized.


In [26]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------------


def load_data(path):
  if os.path.exists(path):
    with open(path, "r", encoding="utf-8") as f:
      return json.load(f)
  return []


def save_checkpoint(path, data):
  # Atomic save ensuring proper UTF-8 Bengali encoding
  temp_path = path + ".tmp"
  with open(temp_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
  os.replace(temp_path, path)


def format_turn_prompt(history, current_role):
  """Formats dialogue history for the API call, ensuring valid user/assistant role sequence and proper context cascading."""
  messages = []

  if current_role == "opponent":
    for turn in history:
      if "for" in turn:
        messages.append({"role": "user", "content": turn["for"]})
      elif "opponent" in turn:
        messages.append({"role": "assistant", "content": turn["opponent"]})
  else:  # current_role == "for"
    # OpenAI/Anthropic APIs require the messages array to start with 'user' role
    messages.append(
        {"role": "user", "content": "বিতর্কের সূচনা ও মূল বিষয়বস্তু:"}
    )
    for turn in history:
      if "for" in turn:
        messages.append({"role": "assistant", "content": turn["for"]})
      elif "opponent" in turn:
        messages.append({"role": "user", "content": turn["opponent"]})

  return messages


async def call_model_api(model_name, messages):
  try:
    response = await client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        seed=SEED,
    )
    return response.choices[0].message.content.strip()
  except Exception as e:
    return f"[API ERROR: {str(e)}]"


async def run_single_debate(prompt_data, for_model, opponent_model):
  prompt_id = prompt_data["id"]
  initial_sentence = prompt_data["sentence"]

  # Turn 1: Provided directly by your dataset input prompt (for)
  first_prompt = f"{initial_sentence}\n\n{COMMON_SUFFIX}"
  dialogue_history = [{"for": first_prompt}]

  roles = ["for", "opponent"]

  # Turns 2 through 16 (15 generated turns total)
  for turn_idx in range(1, TOTAL_TURNS):
    current_role = roles[turn_idx % 2]
    current_model = for_model if current_role == "for" else opponent_model

    # Construct turn-by-turn dialogue history for current speaker
    messages = format_turn_prompt(dialogue_history, current_role)

    # Execute API call and log turn
    response = await call_model_api(current_model, messages)
    dialogue_history.append({current_role: response})

  return {
      "id": prompt_id,
      "for model": for_model,
      "opponent model": opponent_model,
      "Dialogue": dialogue_history,
      "temperature": TEMPERATURE,
      "seed": SEED,
      "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
  }

In [27]:
# ---------------------------------------------------------------------------
# MAIN ASYNC PIPELINE
# ---------------------------------------------------------------------------

def generate_input_json_from_csv(csv_path, output_json_path):
  """Reads 'id' and 'sentence' from the CSV and builds the formatted JSON dataset."""
  if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"CSV file not found at '{csv_path}'. Please upload your CSV to Drive."
    )

  # Read the CSV file
  df = pd.read_csv(csv_path)

  dataset = []
  # Iterate over rows using explicit column names ('id' and 'sentence')
  for _, row in df.iterrows():
    # Retain original ID (converted to string/int safely)
    item_id = row['id']
    sentence_text = str(row['sentence']).strip()

    dataset.append({"id": item_id, "sentence": sentence_text})

  # Save formatted dataset to JSON
  with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

  print(
      f"Created input dataset with {len(dataset)} items at '{output_json_path}'"
  )


In [28]:
# Step 1: Automatically generate bangla_slang_53.json from your CSV file
generate_input_json_from_csv(CSV_FILE, INPUT_FILE)

Created input dataset with 53 items at '/content/drive/MyDrive/SlangPaper/Slang_Dataset/dataset_exp5.json'


In [29]:

async def main():
  os.makedirs(DRIVE_DIR, exist_ok=True)



  # Step 2: Load input dataset and existing results
  prompts = load_data(INPUT_FILE)
  existing_results = load_data(OUTPUT_FILE)

  # Map already completed items for blackout resilience
  completed_keys = {
      (item["id"], item["for model"], item["opponent model"])
      for item in existing_results
  }

  # Generate 20 pairwise model combinations (5 × 4)
  model_pairs = list(permutations(MODELS, 2))

  # Build list of uncompleted tasks
  tasks_to_run = []
  for p in prompts:
    for for_m, opp_m in model_pairs:
      if (p["id"], for_m, opp_m) not in completed_keys:
        tasks_to_run.append((p, for_m, opp_m))

  print(
      f"Total prompts: {len(prompts)} | Total pairs per prompt:"
      f" {len(model_pairs)}"
  )
  print(f"Remaining debates to process: {len(tasks_to_run)}")

  sem = asyncio.Semaphore(CONCURRENT_LIMIT)

  async def process_task(p, f_m, o_m):
    async with sem:
      res = await run_single_debate(p, f_m, o_m)
      existing_results.append(res)
      # Checkpoint to Google Drive immediately after each debate completes
      save_checkpoint(OUTPUT_FILE, existing_results)
      return res

  if tasks_to_run:
    # Wrap tasks into explicit Task objects to allow instant cancellation on Stop
    task_objects = [
        asyncio.create_task(process_task(p, f_m, o_m))
        for p, f_m, o_m in tasks_to_run
    ]

    try:
      await tqdm_asyncio.gather(*task_objects)
    except (KeyboardInterrupt, asyncio.CancelledError):
      print("\n🛑 Stop button clicked! Cancelling running tasks gracefully...")
      for t in task_objects:
        if not t.done():
          t.cancel()

      # Wait for pending cancellations to finish
      await asyncio.gather(*task_objects, return_exceptions=True)
      print("✓ All background calls cancelled. Progress up to this point is saved.")
      return

  print(
      f"\nExperiment execution complete. All output safely stored in:"
      f" {OUTPUT_FILE}"
  )


# Run experiment loop safely
try:
  await main()
except (KeyboardInterrupt, asyncio.CancelledError):
  print("\n🛑 Execution interrupted by user.")

Total prompts: 53 | Total pairs per prompt: 20
Remaining debates to process: 1060


100%|██████████| 1060/1060 [18:06<00:00,  1.03s/it]


Experiment execution complete. All output safely stored in: /content/drive/MyDrive/SlangPaper/Slang_Results/E5_results.json
